# NVIDIA LocateAnything-3B: Visual Grounding Inference Examples

In [ ]:
import json
import boto3

### Query endpoint

In [ ]:
endpoint_name = "!!!name!!!"  # Replace with your endpoint name
region = "!!!region!!!"  # Replace with your endpoint's AWS region


def query_endpoint(payload):
    client = boto3.client("runtime.sagemaker", region_name=region)
    response = client.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="application/json",
        Body=json.dumps(payload).encode("utf-8"),
    )
    result = json.loads(response["Body"].read())
    print(json.dumps(result, indent=2, default=str))
    return result

### Health / Readiness Check

Verify the endpoint is live before running grounding queries. With a trivial prompt and no image, the endpoint returns a readiness status.

In [ ]:
payload = {'inputs': {'prompt': 'ping'}}

In [ ]:
payload = {"inputs": {"prompt": "ping"}}
query_endpoint(payload)

### Open-Set Object Detection

Detect all instances of one or more categories in a scene. Categories are joined and phrased as a detection instruction; the model returns box coordinates for every match. Uses hybrid mode (parallel box decoding with autoregressive fallback).

In [ ]:
payload = {'inputs': {'image': 'iVBORw0KGgoAAAANSUhEUgAAAEAAAABACAIAAAAlC+aJAAAAWklEQVR4nO3PMQ0AMAwDwZTfoWXQBz2ipHfDzp7Zme6+A3jw3AGgQAECAgggQAECCCBAAQIIIIAABQgggAACFCCAAAIIUIAAAgggQAECCCBAAQIIIIAABbwF5wYBQVUq0AAAAABJRU5ErkJggg==',
            'prompt': 'Locate all the instances that matches the following '
                      'description: person</c>car.'},
 'parameters': {'generation_mode': 'hybrid',
                'max_new_tokens': 1024,
                'temperature': 0.0}}

In [ ]:
IMAGE_B64 = "iVBORw0KGgoAAAANSUhEUgAAAEAAAABACAIAAAAlC+aJAAAAWklEQVR4nO3PMQ0AMAwDwZTfoWXQBz2ipHfDzp7Zme6+A3jw3AGgQAECAgggQAECCCBAAQIIIIAABQgggAACFCCAAAIIUIAAAgggQAECCCBAAQIIIIAABbwF5wYBQVUq0AAAAABJRU5ErkJggg=="  # base64 image string (replace with your own)
payload = {
    "inputs": {
        "image": IMAGE_B64,
        "prompt": "Locate all the instances that matches the following description: person</c>car."
    },
    "parameters": {"generation_mode": "hybrid", "max_new_tokens": 1024, "temperature": 0.0}
}
query_endpoint(payload)

### Referring Expression Grounding (Single Instance)

Ground a single object described by a free-form referring phrase. Returns one bounding box for the best-matching region.

In [ ]:
payload = {'inputs': {'image': 'iVBORw0KGgoAAAANSUhEUgAAAEAAAABACAIAAAAlC+aJAAAAWklEQVR4nO3PMQ0AMAwDwZTfoWXQBz2ipHfDzp7Zme6+A3jw3AGgQAECAgggQAECCCBAAQIIIIAABQgggAACFCCAAAIIUIAAAgggQAECCCBAAQIIIIAABbwF5wYBQVUq0AAAAABJRU5ErkJggg==',
            'prompt': 'Locate a single instance that matches the following '
                      'description: the object in the center of the image.'},
 'parameters': {'generation_mode': 'hybrid',
                'max_new_tokens': 256,
                'temperature': 0.0}}

In [ ]:
IMAGE_B64 = "iVBORw0KGgoAAAANSUhEUgAAAEAAAABACAIAAAAlC+aJAAAAWklEQVR4nO3PMQ0AMAwDwZTfoWXQBz2ipHfDzp7Zme6+A3jw3AGgQAECAgggQAECCCBAAQIIIIAABQgggAACFCCAAAIIUIAAAgggQAECCCBAAQIIIIAABbwF5wYBQVUq0AAAAABJRU5ErkJggg=="  # base64 image string (replace with your own)
payload = {
    "inputs": {
        "image": IMAGE_B64,
        "prompt": "Locate a single instance that matches the following description: the object in the center of the image."
    },
    "parameters": {"generation_mode": "hybrid", "max_new_tokens": 256, "temperature": 0.0}
}
query_endpoint(payload)

### Point-Based Localization

Ask the model to point at a target rather than box it. LocateAnything can emit point coordinates (<box> x, y </box>) for fine-grained spatial reasoning, useful for robotics and GUI agents.

In [ ]:
payload = {'inputs': {'image': 'iVBORw0KGgoAAAANSUhEUgAAAEAAAABACAIAAAAlC+aJAAAAWklEQVR4nO3PMQ0AMAwDwZTfoWXQBz2ipHfDzp7Zme6+A3jw3AGgQAECAgggQAECCCBAAQIIIIAABQgggAACFCCAAAIIUIAAAgggQAECCCBAAQIIIIAABbwF5wYBQVUq0AAAAABJRU5ErkJggg==',
            'prompt': 'Point to the main object in the image.'},
 'parameters': {'generation_mode': 'hybrid',
                'max_new_tokens': 128,
                'temperature': 0.0}}

In [ ]:
IMAGE_B64 = "iVBORw0KGgoAAAANSUhEUgAAAEAAAABACAIAAAAlC+aJAAAAWklEQVR4nO3PMQ0AMAwDwZTfoWXQBz2ipHfDzp7Zme6+A3jw3AGgQAECAgggQAECCCBAAQIIIIAABQgggAACFCCAAAIIUIAAAgggQAECCCBAAQIIIIAABbwF5wYBQVUq0AAAAABJRU5ErkJggg=="  # base64 image string (replace with your own)
payload = {
    "inputs": {
        "image": IMAGE_B64,
        "prompt": "Point to the main object in the image."
    },
    "parameters": {"generation_mode": "hybrid", "max_new_tokens": 128, "temperature": 0.0}
}
query_endpoint(payload)

### Fast Mode (Parallel Box Decoding only)

Fast mode uses pure parallel Multi-Token Prediction (MTP) with no autoregressive fallback — highest throughput for clean detection scenes. Compare its speed/output against hybrid mode.

In [ ]:
payload = {'inputs': {'image': 'iVBORw0KGgoAAAANSUhEUgAAAEAAAABACAIAAAAlC+aJAAAAWklEQVR4nO3PMQ0AMAwDwZTfoWXQBz2ipHfDzp7Zme6+A3jw3AGgQAECAgggQAECCCBAAQIIIIAABQgggAACFCCAAAIIUIAAAgggQAECCCBAAQIIIIAABbwF5wYBQVUq0AAAAABJRU5ErkJggg==',
            'prompt': 'Locate all the instances that matches the following '
                      'description: object.'},
 'parameters': {'generation_mode': 'fast',
                'max_new_tokens': 512,
                'temperature': 0.0}}

In [ ]:
IMAGE_B64 = "iVBORw0KGgoAAAANSUhEUgAAAEAAAABACAIAAAAlC+aJAAAAWklEQVR4nO3PMQ0AMAwDwZTfoWXQBz2ipHfDzp7Zme6+A3jw3AGgQAECAgggQAECCCBAAQIIIIAABQgggAACFCCAAAIIUIAAAgggQAECCCBAAQIIIIAABbwF5wYBQVUq0AAAAABJRU5ErkJggg=="  # base64 image string (replace with your own)
payload = {
    "inputs": {
        "image": IMAGE_B64,
        "prompt": "Locate all the instances that matches the following description: object."
    },
    "parameters": {"generation_mode": "fast", "max_new_tokens": 512, "temperature": 0.0}
}
query_endpoint(payload)